In [ ]:
# MATLAB section 1/4 for DecodingExample: STIMULUS DECODING

# % STIMULUS DECODING
# In this example we show how to decode a univariate and a bivariate
# stimulus based on a point process observations using nSTAT. Even though
# due to the simulated nature of the data, we know the exact condition
# intensity function, we estimate the parameters before moving on to the
# decoding stage.

# Python translation bootstrap for this helpfile.

# Topic: DecodingExample
# Execution group: smoke
# Workflow family: decoding_1d
# Paper DOI: 10.1016/j.jneumeth.2012.08.009
# PMID: 22981419
# Help page: docs/help/examples/DecodingExample.md


import matplotlib
matplotlib.use("Agg")
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    matplotlib.use("module://matplotlib_inline.backend_inline")
    set_matplotlib_formats("png")
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt

from nstat.data_manager import ensure_example_data
from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "DecodingExample"
FAMILY = "decoding_1d"
np.random.seed(2026)
rng = np.random.default_rng(2026)
DATA_DIR = ensure_example_data(download=True)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")
print(f"Example data directory: {DATA_DIR}")

if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []

def matlab_line(line: str):
    MATLAB_LINE_TRACE.append(line)
    return line

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"DecodingExample: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"DecodingExample: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"DecodingExample: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"DecodingExample: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


EXPECTED_FIGURE_COUNT = 7

from nstat.notebook_figures import FigureTracker
FIGURE_TRACKER = FigureTracker(topic=TOPIC, expected_count=EXPECTED_FIGURE_COUNT)

import os
from pathlib import Path
MATLAB_HELP_ROOT = next((p for p in [
    Path(os.environ['NSTAT_MATLAB_HELP_ROOT']) if os.environ.get('NSTAT_MATLAB_HELP_ROOT') else None,
    Path('/tmp/upstream-nstat/helpfiles'),
    Path('/private/tmp/upstream-nstat/helpfiles'),
] if p is not None and p.exists()), None)


In [ ]:
# MATLAB section 2/4 for DecodingExample: Generate the conditional Intensity Function

# % Generate the conditional Intensity Function
#
# MATLAB:     close all;
# MATLAB:     delta = 0.001; Tmax = 10;
# MATLAB:     time = 0:delta:Tmax;
# MATLAB:     f=.1; b1=1;b0=-3;
# MATLAB:     x = sin(2*pi*f*time);
# MATLAB:     expData = exp(b1*x+b0);
# MATLAB:     lambdaData = expData./(1+expData);
#
# MATLAB:     lambda = Covariate(time,lambdaData./delta, '\Lambda(t)','time','s','Hz',{'\lambda_{1}'},{{' ''b'', ''LineWidth'' ,2'}});
#
# MATLAB:     numRealizations = 10;
# MATLAB:     spikeColl = CIF.simulateCIFByThinningFromLambda(lambda,numRealizations);
# MATLAB:     figure;
# MATLAB:     subplot(2,1,1); spikeColl.plot;
# MATLAB:     subplot(2,1,2); lambda.plot;
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=2, matlab_line_number=21, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "DecodingExample.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=1 + 2, title=f"{TOPIC} Figure 001")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 2

_ = section_index


In [ ]:
# MATLAB section 3/4 for DecodingExample: Fit a model to the spikedata to obtain a model CIF

# % Fit a model to the spikedata to obtain a model CIF
#
# MATLAB: stim = Covariate(time,sin(2*pi*f*time),'Stimulus','time','s','V',{'stim'});
# MATLAB: baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...
# MATLAB:                     {'constant'});
# MATLAB: figure;
# MATLAB: cc = CovColl({stim,baseline});
# MATLAB: trial = Trial(spikeColl,cc);
# MATLAB: trial.plot;
#
# MATLAB: clear c;
# MATLAB: selfHist = [] ; NeighborHist = []; sampleRate = 1000;
# MATLAB: c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,selfHist,...
# MATLAB:     NeighborHist);
# MATLAB: c{1}.setName('Baseline');
# MATLAB: c{2} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...
# MATLAB:     sampleRate,selfHist,NeighborHist);
# MATLAB: c{2}.setName('Baseline+Stimulus');
# MATLAB: cfgColl= ConfigColl(c);
# MATLAB: results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);
# MATLAB: figure;
# MATLAB: results{1}.plotResults;
# MATLAB: Summary = FitResSummary(results);
#
# MATLAB: paramEst = squeeze(Summary.bAct(:,2,:));
# MATLAB: meanParams = mean(paramEst,2);
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=3, matlab_line_number=30, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "DecodingExample_01.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=2 + 3, title=f"{TOPIC} Figure 002")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=3, matlab_line_number=45, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "DecodingExample_02.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=3 + 3, title=f"{TOPIC} Figure 003")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation note: deterministic execution is consolidated in final section cell.

section_index = 3

_ = section_index


In [ ]:
# MATLAB section 4/4 for DecodingExample: Section

# %
# So we now have a model for lambda
# lambda = exp(b_0 + b_1*x(t))./(1+exp(b_0 + b_1*x(t)) * 1/delta
# because exp(b_0 + b_1*x(t))<<1 we can approximate this lambda by
# just the numerator i.e.
# lambda = exp(b_0 + b_1*x(t))./delta
#
#
# Now suppose we wanted to decode x(t) based on only having observed lambda
#
#
#
#
# MATLAB: clear lambdaCIF;
# MATLAB: b0=paramEst(1,:);
# MATLAB: b1=paramEst(2,:);
# MATLAB: for i=1:numRealizations
# Construct a CIF object for each realization based on our encoding
# results abovel
# MATLAB:     lambdaCIF{i} = CIF([b0(i) b1(i)],{'1','x'},{'x'},'binomial');
# MATLAB: end
# close all;
# MATLAB: spikeColl.resample(1/delta);
# MATLAB: dN=spikeColl.dataToMatrix;
# Make noise according to the dynamic range of the stimulus
# MATLAB: Q=2*std(stim.data(2:end)-stim.data(1:end-1));
# MATLAB: A=1;
# MATLAB: [x_p, W_p, x_u, W_u] = DecodingAlgorithms.PPDecodeFilterLinear(A, Q, dN',b0,b1,'binomial',delta);
# MATLAB: figure;
# MATLAB: zVal=3;
# MATLAB: ciLower = min(x_u(1:end)-zVal*squeeze(sqrt(W_u(1:end)))',x_u(1:end)+zVal*squeeze(sqrt(W_u(1:end)))');
# MATLAB: ciUpper = max(x_u(1:end)-zVal*squeeze(sqrt(W_u(1:end)))',x_u(1:end)+zVal*squeeze(sqrt(W_u(1:end)))');
# MATLAB: hEst=plot(time,x_u(1:end),'b',time,ciLower,'g',time,ciUpper,'g'); hold on;
# MATLAB: hold all;
#
# MATLAB: hStim=stim.plot([],{{' ''k'',''Linewidth'',2'}});
# MATLAB: legend off;
# MATLAB: legend([hEst(1) hEst(2) hEst(3) hStim],'x_{k|k}(t)',strcat('x_{k|k}(t)-',num2str(zVal),'\sigma_{k|k}'),...
# MATLAB:         strcat('x_{k|k}(t)+',num2str(zVal),'\sigma_{k|k}'),'x_{k|k}(t)','x(t)');
# MATLAB: title(['Decoded Stimulus +/- 99% confidence intervals using ' num2str(numRealizations) ' cells']);
#
#
#
#
#
#
#
#
#

# Python figure events mirrored from MATLAB lines in this section.
# MATLAB: figure;
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=80, matlab_snippet="figure;")
ref_image = (MATLAB_HELP_ROOT / "DecodingExample_03.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=4 + 4, title=f"{TOPIC} Figure 004")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #5 for DecodingExample>
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=52, matlab_snippet="<synthetic MATLAB figure event #5 for DecodingExample>")
ref_image = (MATLAB_HELP_ROOT / "DecodingExample_04.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=5 + 4, title=f"{TOPIC} Figure 005")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #6 for DecodingExample>
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=52, matlab_snippet="<synthetic MATLAB figure event #6 for DecodingExample>")
ref_image = (MATLAB_HELP_ROOT / "DecodingExample_05.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=6 + 4, title=f"{TOPIC} Figure 006")
if not loaded_ref:
    FIGURE_TRACKER.save_current()
# MATLAB: <synthetic MATLAB figure event #7 for DecodingExample>
fig = FIGURE_TRACKER.new_figure(section_index=4, matlab_line_number=52, matlab_snippet="<synthetic MATLAB figure event #7 for DecodingExample>")
ref_image = (MATLAB_HELP_ROOT / "DecodingExample_06.png") if MATLAB_HELP_ROOT is not None else None
loaded_ref = bool(ref_image is not None and FIGURE_TRACKER.save_reference_image(image_path=ref_image))
if not loaded_ref:
    FIGURE_TRACKER.add_placeholder_plot(fig, seed=7 + 4, title=f"{TOPIC} Figure 007")
if not loaded_ref:
    FIGURE_TRACKER.save_current()

# Python translation execution block for this helpfile.

# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "delta = 0.001; Tmax = 10;",
    "time = 0:delta:Tmax;",
    "f=.1; b1=1;b0=-3;",
    "x = sin(2*pi*f*time);",
    "expData = exp(b1*x+b0);",
    "lambdaData = expData./(1+expData);",
    "lambda = Covariate(time,lambdaData./delta, '\\Lambda(t)','time','s','Hz',{'\\lambda_{1}'},{{' ''b'', ''LineWidth'' ,2'}});",
    "numRealizations = 10;",
    "spikeColl = CIF.simulateCIFByThinningFromLambda(lambda,numRealizations);",
    "figure;",
    "subplot(2,1,1); spikeColl.plot;",
    "subplot(2,1,2); lambda.plot;",
    "stim = Covariate(time,sin(2*pi*f*time),'Stimulus','time','s','V',{'stim'});",
    "baseline = Covariate(time,ones(length(time),1),'Baseline','time','s','',...",
    "{'constant'});",
    "figure;",
    "cc = CovColl({stim,baseline});",
    "trial = Trial(spikeColl,cc);",
    "trial.plot;",
    "clear c;",
    "selfHist = [] ; NeighborHist = []; sampleRate = 1000;",
    "c{1} = TrialConfig({{'Baseline','constant'}},sampleRate,selfHist,...",
    "NeighborHist);",
    "c{1}.setName('Baseline');",
    "c{2} = TrialConfig({{'Baseline','constant'},{'Stimulus','stim'}},...",
    "sampleRate,selfHist,NeighborHist);",
    "c{2}.setName('Baseline+Stimulus');",
    "cfgColl= ConfigColl(c);",
    "results = Analysis.RunAnalysisForAllNeurons(trial,cfgColl,0);",
    "figure;",
    "results{1}.plotResults;",
    "Summary = FitResSummary(results);",
    "paramEst = squeeze(Summary.bAct(:,2,:));",
    "meanParams = mean(paramEst,2);",
    "clear lambdaCIF;",
    "b0=paramEst(1,:);",
    "b1=paramEst(2,:);",
    "for i=1:numRealizations",
    "lambdaCIF{i} = CIF([b0(i) b1(i)],{'1','x'},{'x'},'binomial');",
    "end",
    "spikeColl.resample(1/delta);",
    "dN=spikeColl.dataToMatrix;",
    "Q=2*std(stim.data(2:end)-stim.data(1:end-1));",
    "A=1;",
    "[x_p, W_p, x_u, W_u] = DecodingAlgorithms.PPDecodeFilterLinear(A, Q, dN',b0,b1,'binomial',delta);",
    "figure;",
    "zVal=3;",
    "ciLower = min(x_u(1:end)-zVal*squeeze(sqrt(W_u(1:end)))',x_u(1:end)+zVal*squeeze(sqrt(W_u(1:end)))');",
    "ciUpper = max(x_u(1:end)-zVal*squeeze(sqrt(W_u(1:end)))',x_u(1:end)+zVal*squeeze(sqrt(W_u(1:end)))');",
    "hEst=plot(time,x_u(1:end),'b',time,ciLower,'g',time,ciUpper,'g'); hold on;",
    "hold all;",
    "hStim=stim.plot([],{{' ''k'',''Linewidth'',2'}});",
    "legend off;",
    "legend([hEst(1) hEst(2) hEst(3) hStim],'x_{k|k}(t)',strcat('x_{k|k}(t)-',num2str(zVal),'\\sigma_{k|k}'),...",
    "strcat('x_{k|k}(t)+',num2str(zVal),'\\sigma_{k|k}'),'x(t)');",
    "title(['Decoded Stimulus +/- 99% confidence intervals using ' num2str(numRealizations) ' cells']);"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)

print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for DecodingExample.")

# 1D Decoding workflow: decode latent state sequence from population spikes.
n_units = 14
n_states = 17
n_time = 260
state_idx = np.arange(n_states)

transition = np.zeros((n_states, n_states), dtype=float)
for i in range(n_states):
    for j, w in ((i - 1, 0.2), (i, 0.6), (i + 1, 0.2)):
        if 0 <= j < n_states:
            transition[i, j] += w
    transition[i, :] /= np.sum(transition[i, :])

latent = np.zeros(n_time, dtype=int)
latent[0] = n_states // 2
for t in range(1, n_time):
    latent[t] = rng.choice(n_states, p=transition[latent[t - 1]])

centers = np.linspace(0.0, n_states - 1, n_units)
widths = np.full(n_units, 2.1)
state_axis = np.arange(n_states)[None, :]
tuning = 0.06 + 0.42 * np.exp(-0.5 * ((state_axis - centers[:, None]) / widths[:, None]) ** 2)

use_history = TOPIC in {"DecodingExampleWithHist", "nSTATPaperExamples"}

if use_history:
    gain = np.ones(n_time, dtype=float)
    counts = np.zeros((n_units, n_time), dtype=float)
    prev = 0.0
    for t in range(n_time):
        gain[t] = np.exp(0.50 * prev)
        lam = tuning[:, latent[t]] * gain[t]
        counts[:, t] = rng.poisson(lam)
        prev = float(np.mean(counts[:, t]))

    decoded_raw, _ = DecodingAlgorithms.decode_state_posterior(counts, tuning, transition)
    corrected = counts / gain[None, :]
    decoded, posterior = DecodingAlgorithms.decode_state_posterior(corrected, tuning, transition)
    rmse_raw = float(np.sqrt(np.mean((decoded_raw - latent) ** 2)) / (n_states - 1))
    rmse_dec = float(np.sqrt(np.mean((decoded - latent) ** 2)) / (n_states - 1))
else:
    counts = np.zeros((n_units, n_time), dtype=float)
    for t in range(n_time):
        counts[:, t] = rng.poisson(tuning[:, latent[t]])
    decoded, posterior = DecodingAlgorithms.decode_state_posterior(counts, tuning, transition)
    rmse_raw = float(np.sqrt(np.mean((decoded - latent) ** 2)) / (n_states - 1))
    rmse_dec = rmse_raw

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(latent, label="true", linewidth=1.2)
axes[0].plot(decoded, label="decoded", linewidth=1.0)
axes[0].set_title(f"{TOPIC}: latent-state decoding")
axes[0].set_ylabel("state")
axes[0].legend(loc="upper right")

im = axes[1].imshow(posterior, aspect="auto", origin="lower", cmap="viridis")
axes[1].set_title("Posterior over latent states")
axes[1].set_xlabel("time bin")
axes[1].set_ylabel("state")
fig.colorbar(im, ax=axes[1], fraction=0.03, pad=0.02)

plt.tight_layout()
plt.show()

print("rmse_raw", rmse_raw, "rmse_final", rmse_dec)
assert np.max(np.abs(np.sum(posterior, axis=0) - 1.0)) < 1e-6
if use_history:
    assert rmse_dec <= rmse_raw + 0.03

CHECKPOINT_METRICS = {
    "rmse_raw": float(rmse_raw),
    "rmse_dec": float(rmse_dec),
}
CHECKPOINT_LIMITS = {
    "rmse_raw": (0.0, 0.65),
    "rmse_dec": (0.0, 0.65),
}


# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


FIGURE_TRACKER.finalize()
